# Pipeline helpers

Data loading, the WEKA-like rule learners (OneR, DecisionTable, JRip), scikit-learn baselines, the per-fold 2-Stage GeS fit (`ges_fit`) and the evaluation metrics. Set `DATA` to the folder holding the CSV files written by `export.R`. Other notebooks load this one with `%run pipeline.ipynb`.

*Part of the code accompanying* **Explainable, Leakage-Free Gene Selection for Breast Cancer Histologic Grade** (Lamba, Munjal, Gigras). Run the notebooks from the folder that holds them, with the exported CSV files in `./data`.

In [ ]:
import numpy as np, pandas as pd, warnings
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score, matthews_corrcoef, balanced_accuracy_score, precision_score, recall_score, confusion_matrix
from imblearn.over_sampling import SMOTEN
%run core.ipynb

warnings.filterwarnings("ignore")
DATA = "./data"
COHORTS = {"GSE7390": "transbig", "UPP": "upp", "MAINZ": "mainz", "UNT": "unt"}

In [ ]:
def load(name, genes=None, zscore=False):
    f = COHORTS[name]
    X = pd.read_csv(f"{DATA}/{f}_expr.csv", index_col=0)
    P = pd.read_csv(f"{DATA}/{f}_pheno.csv", index_col=0)
    P = P.loc[X.index]
    ok = P["grade"].isin([1, 2, 3]).values
    X, P = X[ok], P[ok]
    if genes is not None:
        X = X[genes]
    if zscore:
        X = (X - X.mean()) / X.std(ddof=0).replace(0, 1)
    y = P["grade"].astype(int).values - 1
    return X, y, P

### Simple WEKA-like rule learners

In [ ]:
# ---------------------------------------------------------------- simple WEKA-like rule learners
class OneR:
    def fit(self, D, y, nb, C):
        best = (1e9, 0, None)
        for i in range(D.shape[1]):
            t = np.zeros((nb[i], C)); np.add.at(t, (D[:, i], y), 1)
            err = (t.sum(1) - t.max(1)).sum()
            if err < best[0]:
                best = (err, i, t.argmax(1))
        _, self.i, self.rule = best
        self.default = np.bincount(y, minlength=C).argmax(); self.nb = nb; self.C = C
        return self

    def predict(self, D):
        v = np.minimum(D[:, self.i], self.nb[self.i] - 1)
        return self.rule[v]

In [ ]:
class DecisionTable:
    """WEKA DecisionTable: best-first forward attribute selection by leave-one-out accuracy
    of a majority-class lookup table; unseen patterns fall back to the majority class."""

    def _loo_acc(self, D, y, S):
        if not S:
            return (y == np.bincount(y).argmax()).mean()
        _, inv = np.unique(D[:, S], axis=0, return_inverse=True); inv = inv.ravel()
        cnt = np.zeros((inv.max() + 1, self.C)); np.add.at(cnt, (inv, y), 1)
        c = cnt[inv].copy(); c[np.arange(len(y)), y] -= 1
        pred = np.where(c.sum(1) > 0, c.argmax(1), self.default)
        return (pred == y).mean()

    def fit(self, D, y, nb, C, max_stale=5):
        self.C = C; self.default = np.bincount(y, minlength=C).argmax()
        S, best = [], self._loo_acc(D, y, [])
        stale = 0; cur = []
        while stale < max_stale:
            cand = [(self._loo_acc(D, y, cur + [g]), g) for g in range(D.shape[1]) if g not in cur]
            if not cand: break
            acc, g = max(cand)
            cur = cur + [g]
            if acc > best + 1e-12: best, S, stale = acc, list(cur), 0
            else: stale += 1
        self.S = S
        if S:
            keys, inv = np.unique(D[:, S], axis=0, return_inverse=True); inv = inv.ravel()
            cnt = np.zeros((len(keys), C)); np.add.at(cnt, (inv, y), 1)
            self.table = {tuple(k): cnt[i].argmax() for i, k in enumerate(keys)}
        return self

    def predict(self, D):
        if not self.S: return np.full(len(D), self.default)
        return np.array([self.table.get(tuple(r), self.default) for r in D[:, self.S]])

In [ ]:
class JRip:
    """RIPPER (wittgenstein) applied as WEKA JRip does for multi-class data: classes in
    ascending frequency, one ruleset per class against the remaining ones, majority default."""

    def fit(self, D, y, nb, C):
        import wittgenstein as lw
        self.order = list(np.argsort(np.bincount(y, minlength=C)))
        self.default = self.order[-1]; self.models = []
        df = pd.DataFrame(D, columns=[f"g{i}" for i in range(D.shape[1])]).astype(str)
        keep = np.ones(len(y), bool)
        for c in self.order[:-1]:
            sub = df[keep].copy(); sub["cls"] = (y[keep] == c).astype(int)
            if sub["cls"].sum() == 0: continue
            m = lw.RIPPER(random_state=0, verbosity=0)
            try:
                m.fit(sub, class_feat="cls", pos_class=1)
                self.models.append((c, m))
            except Exception:
                pass
            keep &= (y != c)
        self.cols = df.columns
        return self

    def predict(self, D):
        df = pd.DataFrame(D, columns=self.cols).astype(str)
        pred = np.full(len(D), self.default)
        done = np.zeros(len(D), bool)
        for c, m in self.models:
            try:
                fire = np.asarray(m.predict(df)).astype(bool)
            except Exception:
                fire = np.zeros(len(D), bool)
            new = fire & ~done; pred[new] = c; done |= fire
        return pred

In [ ]:
class SK:
    """sklearn models on one-hot encoded bins (nominal attributes, as WEKA binarises them)."""

    def __init__(self, kind):
        self.kind = kind

    def fit(self, D, y, nb, C):
        self.enc = OneHotEncoder(categories=[list(range(k)) for k in nb], handle_unknown="ignore")
        Z = self.enc.fit_transform(np.minimum(D, np.asarray(nb) - 1)).toarray()
        self.m = {
            "RF": lambda: RandomForestClassifier(n_estimators=100, random_state=1),
            "SVM": lambda: SVC(kernel="linear", C=1.0),
            "IBk": lambda: KNeighborsClassifier(n_neighbors=1, metric="hamming"),
            "DL": lambda: MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=1),
            "LB": lambda: GradientBoostingClassifier(loss="log_loss", n_estimators=10, max_depth=1, random_state=1),
        }[self.kind]()
        self.nb = np.asarray(nb)
        self.m.fit(Z, y)
        return self

    def predict(self, D):
        return self.m.predict(self.enc.transform(np.minimum(D, self.nb - 1)).toarray())

In [ ]:
def make(name):
    return {"HWNB": HNB, "NB": NB, "CWNB": CWNB, "OneR": OneR, "DT": DecisionTable, "JRip": JRip}.get(name, None)


CLASSIFIERS = ["HWNB", "NB", "CWNB", "RF", "SVM", "LB", "IBk", "DL", "JRip", "DT", "OneR"]

In [ ]:
def build(name):
    c = make(name)
    return c() if c is not None else SK(name)

### One training fold

In [ ]:
# ---------------------------------------------------------------- one training fold
def ges_fit(Xtr, ytr, C=3, smote=True, seed=0, stages=("cfs", "cons")):
    """Fit discretisation, SMOTE and 2-Stage GeS on a training fold.
    Returns dict with selected gene indices (into Xtr columns), cut points, discretised
    (augmented) training data restricted to the selected genes."""
    cuts = mdl_cutpoints(Xtr, ytr, C)
    informative = np.array([j for j, c in enumerate(cuts) if len(c) > 0])
    D = apply_cuts(Xtr[:, informative], [cuts[j] for j in informative])
    nb = np.array([len(cuts[j]) + 1 for j in informative])
    Dy = ytr
    if smote:
        sm = SMOTEN(sampling_strategy="not majority", k_neighbors=5, random_state=seed)
        D, Dy = sm.fit_resample(D, ytr)
    res = {"n_informative": len(informative)}
    sel = np.arange(D.shape[1])
    for st in stages:
        if st == "cfs":
            s, merit = cfs_bfs(D[:, sel], Dy, nb[sel])
            res["n_cfs"] = len(s)
        else:
            s, cons, full = consistency_bfs(D[:, sel], Dy, C)
            res["n_cons"] = len(s); res["consistency"] = cons
        sel = sel[s]
    res["genes"] = informative[sel]
    res["cuts"] = [cuts[j] for j in res["genes"]]
    res["nb"] = nb[sel]
    res["Dtr"], res["ytr"] = D[:, sel], Dy
    return res

In [ ]:
def metrics(y, p, C=3):
    cm = confusion_matrix(y, p, labels=list(range(C)))
    fallout = []
    for c in range(C):
        fp = cm[:, c].sum() - cm[c, c]; tn = cm.sum() - cm[c, :].sum() - cm[:, c].sum() + cm[c, c]
        fallout.append(fp / (fp + tn) if fp + tn else 0)
    return {
        "macro_f1": f1_score(y, p, average="macro", labels=list(range(C)), zero_division=0),
        "macro_precision": precision_score(y, p, average="macro", labels=list(range(C)), zero_division=0),
        "macro_recall": recall_score(y, p, average="macro", labels=list(range(C)), zero_division=0),
        "mcc": matthews_corrcoef(y, p),
        "bal_acc": balanced_accuracy_score(y, p),
        "fallout": float(np.mean(fallout)),
        "recall_g": recall_score(y, p, average=None, labels=list(range(C)), zero_division=0),
        "f1_g": f1_score(y, p, average=None, labels=list(range(C)), zero_division=0),
        "fallout_g": np.array(fallout),
    }